# AOI — Train the real defect detector (DsPCBSD+ / YOLOv8) on Colab GPU

Trains the real YOLOv8 defect model that replaces the synthetic inference events.
See `docs/implementation/REAL_INFERENCE_INTEGRATION_PLAN.md` for the full plan.

**Runtime → Change runtime type → GPU (T4 is fine).**

Data-flow principle (important):
- **Dataset lives on Colab local disk** (`/content`) during training — mounted Drive is far too slow for 10k small-file reads.
- **Outputs (weights, runs, eval) live on Drive** via a symlink, so a disconnect never wipes them and `--resume` works.
- The big dataset **zip is cached on Drive** so you only download it once.

Do **not** upload the 10k images from your laptop — Colab pulls them from Kaggle far faster.

In [43]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
import os
os.chdir("/content")

In [46]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/AOI"
os.makedirs(DRIVE, exist_ok=True)
os.chdir(DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
# 1. Confirm GPU + mount Drive (Drive is for OUTPUTS and your own images only)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/AOI'
os.makedirs(f'{DRIVE}/datasets', exist_ok=True)
os.makedirs(f'{DRIVE}/ml_models', exist_ok=True)
print('Drive ready at', DRIVE)

GPU 0: Tesla T4 (UUID: GPU-523f13cb-5a1b-cec5-1d58-72d76d5e6550)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive ready at /content/drive/MyDrive/AOI


In [48]:
# 2. Clone the repo (gives the variant-aware pipeline, not just plain YOLO) + deps
!rm -rf /content/AOI
!git clone https://github.com/lystiger/AOI.git /content/AOI
%cd /content/AOI
!pip install -q ultralytics kaggle

# Persist all training outputs to Drive so a disconnect can't wipe them and --resume works.
# (Only outputs go through Drive; the dataset stays on fast local disk below.)
!rm -rf /content/AOI/ml/models
!ln -s /content/drive/MyDrive/AOI/ml_models /content/AOI/ml/models
!ls -la /content/AOI/ml/models

Cloning into '/content/AOI'...
remote: Enumerating objects: 1436, done.
remote: Counting objects: 100% (446/446), done.
remote: Compressing objects: 100% (313/313), done.
remote: Total 1436 (delta 157), reused 380 (delta 107), pack-reused 990 (from 1)
Receiving objects: 100% (1436/1436), 69.51 MiB | 18.79 MiB/s, done.
Resolving deltas: 100% (736/736), done.
/content/AOI
lrwxrwxrwx 1 root root 36 Jun 29 16:24 /content/AOI/ml/models -> /content/drive/MyDrive/AOI/ml_models


### Kaggle token
Upload your `kaggle.json` (Kaggle → Account → Create New API Token) to `MyDrive/AOI/kaggle.json` once.
If the dataset slug 404s, find the current one with `!kaggle datasets list -s dspcbsd`.

In [49]:
# 3. Get DsPCBSD+ onto LOCAL disk (cache the zip on Drive to avoid re-downloading)
import os, glob, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy2(f'{DRIVE}/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

zip_cache = f'{DRIVE}/datasets/dspcbsd_plus.zip'
if not os.path.exists(zip_cache):
    !kaggle datasets download -d enisteper1/dataset-of-pcb-surface-defects-dspcbsd -p /content/dl
    shutil.copy2(glob.glob('/content/dl/*.zip')[0], zip_cache)
    print('Cached zip to Drive.')
else:
    print('Using cached zip from Drive.')

!mkdir -p /content/data/dspcbsd_raw
!unzip -q -o "{zip_cache}" -d /content/data/dspcbsd_raw
print('Top-level contents of the download:')
!find /content/data/dspcbsd_raw -maxdepth 2 | head -40

Using cached zip from Drive.
Top-level contents of the download:
/content/data/dspcbsd_raw
/content/data/dspcbsd_raw/DsPCBSD+
/content/data/dspcbsd_raw/DsPCBSD+/data.yaml
/content/data/dspcbsd_raw/DsPCBSD+/Data_YOLO


In [50]:
# 4. Normalize whatever layout the download has -> train/val/test + data.yaml (LOCAL disk)
!python -m ml.pipeline.dspcbsd_dataset \
    --source-root /content/data/dspcbsd_raw \
    --output-root /content/AOI/ml/data/dspcbsd_plus \
    --overwrite
print('\n--- data.yaml ---')
print(open('/content/AOI/ml/data/dspcbsd_plus/data.yaml').read())

Class names from DsPCBSD+/data.yaml: ['SH', 'SP', 'SC', 'OP', 'MB', 'HB', 'CS', 'CFO', 'BMFO']
{
  "source_root": "/content/data/dspcbsd_raw",
  "output_root": "/content/AOI/ml/data/dspcbsd_plus",
  "seed": 42,
  "classes": [
    "SH",
    "SP",
    "SC",
    "OP",
    "MB",
    "HB",
    "CS",
    "CFO",
    "BMFO"
  ],
  "split_image_counts": {
    "train": 7388,
    "val": 2051,
    "test": 820
  },
  "split_box_counts": {
    "train": 14592,
    "val": 4092,
    "test": 1592
  },
  "class_counts": {
    "BMFO": 1680,
    "CFO": 1832,
    "CS": 2490,
    "HB": 2883,
    "MB": 2529,
    "OP": 1770,
    "SC": 1593,
    "SH": 915,
    "SP": 4584
  },
  "negative_images": {},
  "images_missing_labels": 0,
  "report_path": "/content/AOI/ml/reports/defect_detection/20260629-162413Z-dataset.md"
}

--- data.yaml ---
path: /content/AOI/ml/data/dspcbsd_plus
train: train/images
val: val/images
test: test/images

nc: 9
names:
  0: SH
  1: SP
  2: SC
  3: OP
  4: MB
  5: HB
  6: CS
  7: CFO
  8:

### Training notes
- Repo defaults (`imgsz=1280, epochs=100, batch=8`) will OOM/time out a free T4. Start with `imgsz=640, epochs=60, batch=16`; raise `imgsz` later if the tiny trace defects need it.
- Outputs land under `ml/models/component_detection/` → which is symlinked to Drive, so they persist.
- If Colab disconnects mid-run, re-run the same cell with `--resume` appended to continue from `last.pt`.

In [51]:
# 5a. Train the baseline YOLOv8s defect detector
!python -m ml.pipeline.component_train \
    --variant baseline \
    --dataset-root /content/AOI/ml/data/dspcbsd_plus \
    --imgsz 640 --epochs 60 --batch 16 --device 0

Dataset: /content/AOI/ml/data/dspcbsd_plus/data.yaml
  train: 7388 images, 7388 labels
  val: 2051 images, 2051 labels
  test: 820 images, 820 labels
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/AOI/ml/data/dspcbsd_plus/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=Fa

In [52]:
# 5b. Train the channel-attention variant (feeds the S09 degradation comparison)
!python -m ml.pipeline.component_train \
    --variant channel_attention \
    --dataset-root /content/AOI/ml/data/dspcbsd_plus \
    --imgsz 640 --epochs 60 --batch 16 --device 0

Dataset: /content/AOI/ml/data/dspcbsd_plus/data.yaml
  train: 7388 images, 7388 labels
  val: 2051 images, 2051 labels
  test: 820 images, 820 labels
Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/AOI/ml/pipeline/component_train.py", line 219, in <module>
    main()
  File "/content/AOI/ml/pipeline/component_train.py", line 205, in main
    train_component_model(
  File "/content/AOI/ml/pipeline/component_train.py", line 107, in train_component_model
    results = model.train(
              ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 801, in train
    self.trainer = (trainer or self._smart_load("trainer"))(overrides=args, _callbacks=self.callbacks)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [53]:
# 6. Evaluate on the held-out test split -> honest mAP/precision/recall + confusion matrix
!python -m ml.pipeline.component_evaluate \
    --variant baseline --split test \
    --dataset-root /content/AOI/ml/data/dspcbsd_plus \
    --imgsz 640

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/AOI/ml/pipeline/component_evaluate.py", line 289, in <module>
    main()
  File "/content/AOI/ml/pipeline/component_evaluate.py", line 276, in main
    evaluate_component_model(
  File "/content/AOI/ml/pipeline/component_evaluate.py", line 31, in evaluate_component_model
    weights = weights_path or get_component_weights_for_variant(variant)
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AOI/ml/pipeline/component_train.py", line 185, in get_component_weights_for_variant
    return get_latest_component_weights()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AOI/ml/pipeline/component_train.py", line 179, in get_latest_component_weights
    raise FileNotFoundError(f"No trained component weights found at {weights}. Run train_component_model() first.")
FileNotFoundError: No traine

In [54]:
# 7. Weights already persist on Drive (via the symlink). Confirm what landed.
!ls -la /content/drive/MyDrive/AOI/ml_models/component_detection/*.pt 2>/dev/null || echo 'no top-level weights yet'
!find /content/drive/MyDrive/AOI/ml_models -name 'best*.pt' | head

no top-level weights yet
/content/drive/MyDrive/AOI/ml_models/component_detection/component_detection/pcb_seed_v1/weights/best.pt


## Next steps
1. Download `best-baseline.pt` (and `best.pt`) from `MyDrive/AOI/ml_models/component_detection/` back to your laptop into `ml/models/defect_detection/`.
2. Back in the app, the wired `inference_runner` (Workstream 3) loads it and POSTs real events into the Loki/Grafana pipeline.
3. Run the experiment suite against the real model per the test matrix in the plan.